In [2]:
%load_ext autoreload
%autoreload 2

import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parents[1]))

import numpy as np
import pandas as pd
from src.utils.pipeline import load_all_snapshots
from src.data.validation import plate_appearances
from src.models.marcel import season_lines
from src.models.projection import (
    SKILL_FEATURES, build_dataset, cross_validated_mae, skill_rates,
)

print("ready")

ready


In [3]:
# Marcel needs three prior seasons; skills need one. Start at 2018 so
# every input season is inside the Statcast era — Day 2 established that
# the 2015 tracking change makes earlier seasons non-comparable for
# skill rates.
#
# 2020 is excluded: a 60-game season distorts both its own rates and the
# Marcel windows it sits in.
TARGETS = [2018, 2019, 2021, 2022, 2023, 2024]

df = load_all_snapshots(seasons=list(range(2014, 2025)))
df["season"] = pd.to_datetime(df["game_date"]).dt.year
pa = plate_appearances(df)
pitcher_ids = set(df["pitcher"].dropna().unique())

lines = season_lines(pa, pitcher_ids=pitcher_ids)
skills = skill_rates(df, pitcher_ids=pitcher_ids)
print(f"{len(lines)} batter-seasons, {len(skills)} skill rows")
print(lines.groupby("season").size().to_string())

5316 batter-seasons, 5316 skill rows
season
2014    503
2015    489
2016    485
2017    463
2018    452
2019    451
2020    417
2021    481
2022    525
2023    512
2024    538


In [4]:
rows = []
for target in TARGETS:
    try:
        data = build_dataset(lines, skills, target)
    except Exception as exc:
        print(f"{target}: {exc}")
        continue
    if len(data) < 100:
        print(f"{target}: only {len(data)} batters, skipping")
        continue

    for metric in ["k", "bb"]:
        res = cross_validated_mae(data, metric)
        rows.append({
            "season": target,
            "metric": f"{metric.upper()}%",
            "n": len(data),
            "marcel": res["marcel"],
            "rescaled": res["marcel_rescaled"],
            "plus_skills": res["marcel_plus_skills"],
        })

r = pd.DataFrame(rows)
r["gain_vs_marcel"] = (r["marcel"] - r["plus_skills"]) / r["marcel"]
print(r.round(4).to_string(index=False))

 season metric   n  marcel  rescaled  plus_skills  gain_vs_marcel
   2018     K% 205  0.0249    0.0253       0.0257         -0.0296
   2018    BB% 205  0.0155    0.0155       0.0155         -0.0053
   2019     K% 206  0.0273    0.0262       0.0262          0.0426
   2019    BB% 206  0.0152    0.0155       0.0157         -0.0351
   2021     K% 193  0.0263    0.0264       0.0261          0.0087
   2021    BB% 193  0.0153    0.0160       0.0157         -0.0260
   2022     K% 214  0.0311    0.0306       0.0298          0.0413
   2022    BB% 214  0.0163    0.0152       0.0151          0.0727
   2023     K% 227  0.0276    0.0274       0.0269          0.0268
   2023    BB% 227  0.0145    0.0147       0.0145          0.0058
   2024     K% 249  0.0271    0.0269       0.0264          0.0244
   2024    BB% 249  0.0144    0.0143       0.0136          0.0555


In [5]:
print("Marcel's own MAE by season:")
print(r.pivot(index="season", columns="metric", values="marcel").round(4).to_string())
print()
for metric in ["K%", "BB%"]:
    g = r[r["metric"] == metric]["gain_vs_marcel"]
    print(f"{metric}: mean {g.mean():+.2%}, median {g.median():+.2%}, "
          f"range {g.min():+.2%} to {g.max():+.2%}, "
          f"positive {(g > 0).sum()}/{len(g)}")

Marcel's own MAE by season:
metric     BB%      K%
season                
2018    0.0155  0.0249
2019    0.0152  0.0273
2021    0.0153  0.0263
2022    0.0163  0.0311
2023    0.0145  0.0276
2024    0.0144  0.0271

K%: mean +1.90%, median +2.56%, range -2.96% to +4.26%, positive 5/6
BB%: mean +1.13%, median +0.02%, range -3.51% to +7.27%, positive 3/6
